# NFS Manifest to S3 and BigQuery Index

This notebook inspects the local repo tree, builds a lightweight manifest, and reshapes that manifest into records that could be emitted to object storage and queried through BigQuery or BigLake-style external tables.

The local repo remains authoritative. The generated manifest is the search-friendly derivative.

In [ ]:
from collections import Counter
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path

def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    if cwd.name == "notebooks":
        return cwd.parent
    return cwd

REPO_ROOT = resolve_repo_root()
REPO_ROOT

In [ ]:
def iter_indexable_files(root: Path):
    ignored_parts = {".git", "__pycache__", ".pytest_cache"}
    ignored_names = {".env"}
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        if any(part in ignored_parts for part in path.parts):
            continue
        if path.name in ignored_names:
            continue
        if path.stat().st_size > 2_000_000:
            continue
        yield path

def digest_path(path: Path) -> str:
    data = path.read_bytes()
    return sha256(data).hexdigest()

manifest = []
timestamp = datetime.now(timezone.utc).isoformat()

for path in iter_indexable_files(REPO_ROOT):
    rel = path.relative_to(REPO_ROOT)
    manifest.append({
        "path": rel.as_posix(),
        "suffix": path.suffix or "<none>",
        "size_bytes": path.stat().st_size,
        "sha256": digest_path(path),
        "indexed_at": timestamp,
        "top_level": rel.parts[0] if rel.parts else ".",
    })

len(manifest)

In [ ]:
suffix_counts = Counter(row["suffix"] for row in manifest)
top_level_counts = Counter(row["top_level"] for row in manifest)

print("Top suffixes:")
for suffix, count in suffix_counts.most_common(10):
    print(f"  {suffix:12} {count}")

print("\nTop components:")
for component, count in top_level_counts.most_common(10):
    print(f"  {component:12} {count}")

print("\nSample rows:")
for row in manifest[:5]:
    print(row)

In [ ]:
for row in manifest:
    row["bucket"] = "new-horizons-index"
    row["partition_date"] = row["indexed_at"][:10]
    row["s3_key"] = (
        f"manifests/date={row['partition_date']}/component={row['top_level']}/"
        f"{row['sha256'][:12]}.json"
    )

for row in manifest[:5]:
    print({
        "path": row["path"],
        "s3_uri": f"s3://{row['bucket']}/{row['s3_key']}",
    })

In [ ]:
bigquery_ddl = f"""
CREATE OR REPLACE EXTERNAL TABLE `new_horizons.repo_manifest`
(
  path STRING,
  suffix STRING,
  size_bytes INT64,
  sha256 STRING,
  indexed_at TIMESTAMP,
  top_level STRING,
  bucket STRING,
  partition_date DATE,
  s3_key STRING
)
OPTIONS (
  format = 'PARQUET',
  uris = ['s3://new-horizons-index/manifests/*']
);
""".strip()

print(bigquery_ddl)

In [ ]:
def search_manifest(term: str, rows: list[dict]) -> list[dict]:
    term = term.lower()
    return [row for row in rows if term in row["path"].lower()]

results = search_manifest("readme", manifest)
print(f"Found {len(results)} rows matching 'readme'\n")
for row in results[:10]:
    print(row["path"], row["size_bytes"], row["sha256"][:12])

## Takeaway

This sketch keeps the repo tree local and easy to inspect, then emits a manifest layer that is cheap to store in object storage and easy to query in SQL. That is the intended split between the canonical workspace and the analytics-friendly index.